In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import spreg
import esda
import libpysal
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial
from scipy import stats
from config.config import DATA_INTERIM, OUTPUTS


In [ ]:
warnings.filterwarnings("ignore")

os.makedirs(OUTPUTS / "tables", exist_ok=True)
os.makedirs(OUTPUTS / "figures", exist_ok=True)

# Modelling

Replicates the analytic strategy of Suss & Oliveira (2023), using 2025 S&S data and updated control variables. The analysis proceeds in two stages following the paper:

1. **OLS and Spatial Durbin Model (SDM)** - treats stops as a continuous variable, tests for and controls spatial autocorrelation using Queen contiguity weights and spatial lags of outcome and regressors.
2. **Negative binomial regression** - accounts for the count nature and overdispersion of the S&S outcome variable. Three models: Gini only (Model 1), full controls (Model 2) and Gini x income deprivation interaction (Model 3).

All independent variables are standardised (mean = 0, SD = 1) prior to modelling, following the paper. Borough fixed effects are included in all models.

Moran's I is computed on OLS residuals to confirm spatial autocorrelation and on SDM residuals to confirm it is eliminated, following the paper's analytic strategy (Table 2 and associated text).

## Load and prepare data

In [ ]:
# Load analytical dataset
analytical = pd.read_csv(DATA_INTERIM / "analytical_dataset.csv")
print(f"Full dataset: {len(analytical)} LSOAs")

# Load LSOA boundaries for spatial weights
london_gdf = gpd.read_file(
    DATA_INTERIM / "boundaries" / "london_lsoa_2021.gpkg",
    layer="london_lsoa_2021"
)[["LSOA21CD", "geometry"]]

## Prepare regression sample
Drop LSOAs with missing values on any model variable. The main source of attrition is the Gini threshold (n=30 transactions required). LSOAs with no S&S activity (zero stops) are retained as the paper includes them.

In [ ]:
model_vars = [
    "LSOA21CD", "LAD22NM", "ss_count", "ss_rate_per1000",
    "gini_housing", "drug_rate_2024", "income_score",
    "imd_crime_score", "pct_non_white", "mean_dist_to_tfl_m",
    "mean_price", "pop_density"
]

reg_df = analytical[model_vars].dropna().copy()
print(f"Regression sample: {len(reg_df)} LSOAs "
      f"({len(reg_df)/len(analytical)*100:.1f}% of full dataset)")
print(f"Dropped: {len(analytical) - len(reg_df)} LSOAs")

## Variable transformations
Following the paper:
- 'pop_density' and 'mean_price' are logged before standardisation
- All independent variables are standardised (mean = 0, SD = 1)
- 'ss_rate_per1000' is logged for the robustness check OLS/SDM models

In [ ]:
reg_df["log_density"] = np.log(reg_df["pop_density"])
reg_df["log_mean_price"] = np.log(reg_df["mean_price"])
reg_df["log_ss_rate"] = np.log(reg_df["ss_rate_per1000"].clip(lower=0.001))

# Variables to standardise
to_standardise = [
    "gini_housing", "log_mean_price", "log_density",
    "income_score", "imd_crime_score", "drug_rate_2024",
    "pct_non_white", "mean_dist_to_tfl_m", "log_ss_rate"
]

for col in to_standardise:
    mean = reg_df[col].mean()
    std = reg_df[col].std()
    reg_df[f"{col}_std"] = (reg_df[col] - mean) / std

print("Standardised variables created")
print(f"\nDescriptive statistics (standardised):")
std_cols = [f"{c}_std" for c in to_standardise]
print(reg_df[std_cols].describe().round(3).to_string())

## Borough fixed effects
Borough dummies created via one-hot encoding with one borough dropped as the reference category (alphabetically first: Barking and Dagenham).

In [ ]:
borough_dummies = pd.get_dummies(
    reg_df["LAD22NM"], drop_first=True, dtype=float
)
print(f"Borough dummies: {borough_dummies.shape[1]} columns "
      f"(reference: {reg_df['LAD22NM'].sort_values().iloc[0]})")

## Spatial weights matrix
Queen contiguity weights, row-standardised, built from the LSOA boundaries. Only LSOAs in the regression sample are included.
The Queen configuration assigns non-zero weights to all LSOAs sharing a boundary or corner, following the paper.

In [ ]:
# Filter boundaries to regression sample and sort to match reg_df order
reg_gdf = london_gdf.merge(reg_df[["LSOA21CD"]], on="LSOA21CD", how="inner")
reg_gdf = reg_gdf.set_index("LSOA21CD").loc[reg_df["LSOA21CD"]].reset_index()

print(f"Building Queen contiguity weights for {len(reg_gdf)} LSOAs...")
w = libpysal.weights.Queen.from_dataframe(reg_gdf, geom_col="geometry")
w.transform = "r"  # Row-standardise
print(f"Spatial weights built: {w.n} units, mean neighbours: {w.mean_neighbors:.2f}")

## Build design matrices

Outcome variable: 'ss_count' (raw count, for OLS and negative binomial)
Independent variables: standardised Gini + all standardised controls + borough fixed effects

In [ ]:
X_cols = [
    "gini_housing_std",
    "log_mean_price_std",
    "log_density_std",
    "income_score_std",
    "imd_crime_score_std",
    "drug_rate_2024_std",
    "pct_non_white_std",
    "mean_dist_to_tfl_m_std"
]

X_names = [
    "Gini",
    "Average property value (log)",
    "Density (log)",
    "Income deprivation",
    "Crime deprivation",
    "Drugs rate",
    "Non-white (%)",
    "TfL station distance"
]

y = reg_df["ss_count"].values
X_base = reg_df[X_cols].values
X_with_fe = np.hstack([X_base, borough_dummies.loc[reg_df.index].values])

# For statsmodels (needs constant)
X_sm = sm.add_constant(
    np.hstack([X_base, borough_dummies.loc[reg_df.index].values])
)

print(f"y shape: {y.shape}")
print(f"X shape (with FE): {X_with_fe.shape}")

## Table 2: OLS and Spatial Durbin Model
OLS treats stops as a continuous variable and serves as a baseline.
Moran's I on OLS residuals tests for spatial autocorrelation.
The SDM adds spatial lags of the outcome (rho) and all regressors (lambda), using Queen continuity weights. Moran's I on SDM residuals confirms autocorrelation is eliminated.

In [ ]:
# --- OLS ---
print("Fitting OLS...")
ols = spreg.OLS(
    y.reshape(-1, 1),
    X_with_fe,
    w=w,
    spat_diag=True,
    moran=True,
    name_y="ss_count",
    name_x=X_names + list(borough_dummies.columns)
)

print(f"OLS R²: {ols.r2:.3f}")
print(f"OLS Moran's I on residuals: {ols.moran_res[0]:.3f} "
      f"(p={ols.moran_res[2]:.4f})")

In [ ]:
# --- SDM (ML_Lag with slx_lags=1) ---
print("Fitting Spatial Durbin Model...")
sdm = spreg.ML_Lag(
    y.reshape(-1, 1),
    X_with_fe,
    w=w,
    slx_lags=1,
    spat_diag=True,
    name_y="ss_count",
    name_x=X_names + list(borough_dummies.columns)
)

print(f"SDM Log-likelihood: {sdm.logll:.3f}")
print(f"SDM rho (spatial lag of y): {sdm.rho:.3f}")

# Moran's I on SDM residuals via Monte Carlo simulation (1,000 permutations)
print("Running Monte Carlo Moran's I on SDM residuals (1,000 permutations)...")
mi_sdm = esda.Moran(sdm.u.flatten(), w, permutations=1000)
print(f"SDM Moran's I on residuals: {mi_sdm.I:.3f} (p={mi_sdm.p_sim:.3f})")

In [ ]:
# --- Build Table 2 ---
def build_table2(ols_model, sdm_model, x_names):
    """Extract OLS and SDM coefficients for the main regressors."""
    n_main = len(x_names)

    rows = []
    for i, name in enumerate(x_names):
        ols_coef = ols_model.betas[i + 1][0]  # +1 for constant
        ols_se = ols_model.std_err[i + 1]
        ols_t = ols_model.t_stat[i + 1][0]
        ols_p = 2 * (1 - stats.t.cdf(abs(ols_t), df=ols_model.n - ols_model.k))

        sdm_coef = sdm_model.betas[i][0]
        sdm_se = sdm_model.std_err[i]
        sdm_p = sdm_model.z_stat[i][1]

        def stars(p):
            if p < 0.01: return "***"
            elif p < 0.05: return "**"
            elif p < 0.1: return "*"
            return ""

        rows.append({
            "Variable": name,
            "OLS coef": f"{ols_coef:.3f}{stars(ols_p)}",
            "OLS SE": f"({ols_se:.3f})",
            "SDM coef": f"{sdm_coef:.3f}{stars(sdm_p)}",
            "SDM SE": f"({sdm_se:.3f})"
        })

    # Model stats
    rows.append({"Variable": "Borough fixed effects",
                 "OLS coef": "Y", "OLS SE": "",
                 "SDM coef": "Y", "SDM SE": ""})
    rows.append({"Variable": "Rho",
                 "OLS coef": "",  "OLS SE": "",
                 "SDM coef": f"{sdm_model.rho:.3f}", "SDM SE": ""})
    rows.append({"Variable": "Observations",
                 "OLS coef": str(ols_model.n), "OLS SE": "",
                 "SDM coef": str(sdm_model.n), "SDM SE": ""})
    rows.append({"Variable": "R²",
                 "OLS coef": f"{ols_model.r2:.3f}", "OLS SE": "",
                 "SDM coef": "", "SDM SE": ""})
    rows.append({"Variable": "Log-likelihood",
                 "OLS coef": "", "OLS SE": "",
                 "SDM coef": f"{sdm_model.logll:.3f}", "SDM SE": ""})

    return pd.DataFrame(rows)

table2 = build_table2(ols, sdm, X_names)
print("\nTable 2: OLS and SDM results")
print(table2.to_string(index=False))

table2.to_csv(OUTPUTS / "tables" / "table2_ols_sdm.csv", index=False)
print("\nSaved table2_ols_sdm.csv")

## Table 3: Negative Binomial Regression

The negative binomial model is appropriate for count data with overdispersion (variance exceeds mean). Three models following the paper:
- Model 1: Gini coefficient only + borough fixed effects
- Model 2: Full controls + borough fixed effects (main model)
- Model 3: Model 2 + Gini x income deprivation interaction term

In [ ]:
borough_fe = borough_dummies.loc[reg_df.index].values
gini_std = reg_df["gini_housing_std"].values
income_std = reg_df["income_score_std"].values

# --- Model 1: Gini only ---
print("Fitting Negative Binomial Model 1 (Gini only)...")
X_nb1 = sm.add_constant(
    np.hstack([gini_std.reshape(-1, 1), borough_fe])
)
nb1 = NegativeBinomial(y, X_nb1).fit(
    method="bfgs", maxiter=1000, disp=False
)
print(f"Model 1 Log-likelihood: {nb1.llf:.3f}")
print(f"Model 1 Gini coef: {nb1.params[1]:.3f} "
      f"(p={nb1.pvalues[1]:.4f})")

In [ ]:
# --- Model 2: Full controls ---
print("Fitting Negative Binomial Model 2 (full controls)...")
X_nb2 = sm.add_constant(np.hstack([X_base, borough_fe]))
nb2 = NegativeBinomial(y, X_nb2).fit(
    method="bfgs", maxiter=1000, disp=False
)
print(f"Model 2 Log-likelihood: {nb2.llf:.3f}")
print(f"Model 2 Gini coef: {nb2.params[1]:.3f} "
      f"(p={nb2.pvalues[1]:.4f})")
print(f"Model 2 Gini IRR: {np.exp(nb2.params[1]):.3f}")

In [ ]:
# --- Model 3: Interaction term ---
print("Fitting Negative Binomial Model 3 (with interaction)...")
interaction = (gini_std * income_std).reshape(-1, 1)
X_nb3 = sm.add_constant(
    np.hstack([X_base, interaction, borough_fe])
)
nb3 = NegativeBinomial(y, X_nb3).fit(
    method="bfgs", maxiter=1000, disp=False
)
print(f"Model 3 Log-likelihood: {nb3.llf:.3f}")
print(f"Model 3 Gini coef: {nb3.params[1]:.3f} "
      f"(p={nb3.pvalues[1]:.4f})")
print(f"Model 3 Gini × Income interaction: {nb3.params[9]:.3f} "
      f"(p={nb3.pvalues[9]:.4f})")

In [ ]:
# --- Build Table 3 ---
def build_table3(nb1, nb2, nb3, x_names):
    """Extract negative binomial coefficients for main regressors."""
    def stars(p):
        if p < 0.01: return "***"
        elif p < 0.05: return "**"
        elif p < 0.1: return "*"
        return ""

    rows = []

    # Gini (index 1 in all models, after constant)
    for i, name in enumerate(x_names):
        idx2 = i + 1  # +1 for constant
        idx3 = i + 1

        if i == 0:  # Gini
            m1_coef = f"{nb1.params[1]:.3f}{stars(nb1.pvalues[1])}"
            m1_se = f"({nb1.bse[1]:.3f})"
        else:
            m1_coef = ""
            m1_se = ""

        m2_coef = f"{nb2.params[idx2]:.3f}{stars(nb2.pvalues[idx2])}"
        m2_se = f"({nb2.bse[idx2]:.3f})"
        m3_coef = f"{nb3.params[idx3]:.3f}{stars(nb3.pvalues[idx3])}"
        m3_se = f"({nb3.bse[idx3]:.3f})"

        rows.append({
            "Variable": name,
            "Model 1 coef": m1_coef, "Model 1 SE": m1_se,
            "Model 2 coef": m2_coef, "Model 2 SE": m2_se,
            "Model 3 coef": m3_coef, "Model 3 SE": m3_se
        })

    # Interaction term (Model 3 only, index 9)
    rows.append({
        "Variable": "Gini × Income deprivation",
        "Model 1 coef": "", "Model 1 SE": "",
        "Model 2 coef": "", "Model 2 SE": "",
        "Model 3 coef": f"{nb3.params[9]:.3f}{stars(nb3.pvalues[9])}",
        "Model 3 SE": f"({nb3.bse[9]:.3f})"
    })

    # Model stats
    rows.append({"Variable": "Borough fixed effects",
                 "Model 1 coef": "Y", "Model 1 SE": "",
                 "Model 2 coef": "Y", "Model 2 SE": "",
                 "Model 3 coef": "Y", "Model 3 SE": ""})
    rows.append({"Variable": "Observations",
                 "Model 1 coef": str(int(nb1.nobs)),
                 "Model 1 SE": "",
                 "Model 2 coef": str(int(nb2.nobs)),
                 "Model 2 SE": "",
                 "Model 3 coef": str(int(nb3.nobs)),
                 "Model 3 SE": ""})
    rows.append({"Variable": "Log-likelihood",
                 "Model 1 coef": f"{nb1.llf:.3f}", "Model 1 SE": "",
                 "Model 2 coef": f"{nb2.llf:.3f}", "Model 2 SE": "",
                 "Model 3 coef": f"{nb3.llf:.3f}", "Model 3 SE": ""})
    rows.append({"Variable": "AIC",
                 "Model 1 coef": f"{nb1.aic:.3f}", "Model 1 SE": "",
                 "Model 2 coef": f"{nb2.aic:.3f}", "Model 2 SE": "",
                 "Model 3 coef": f"{nb3.aic:.3f}", "Model 3 SE": ""})

    return pd.DataFrame(rows)

table3 = build_table3(nb1, nb2, nb3, X_names)
print("\nTable 3: Negative Binomial results")
print(table3.to_string(index=False))

table3.to_csv(OUTPUTS / "tables" / "table3_negative_binomial.csv", index=False)
print("\nSaved table3_negative_binomial.csv")

## Figure 3: Marginal effects plot

Replicates Figure 3 from the paper - marginal effect and average partial effect for each covariate from Model 2 of Table 3, with 95% confidence intervals.

In [ ]:
# Marginal effect at the mean (MEM) — predicted count when all other
# variables are at their mean (i.e. zero after standardisation)
# APE — average over all observations

def marginal_effects_nb(model, X, x_names, n_main):
    """
    Calculate marginal effects and average partial effects for a
    negative binomial model. Returns a DataFrame with MEM and APE
    for the first n_main variables (excluding constant and FE dummies).
    """
    betas = model.params[:-1]
    # Predicted mean for each observation
    mu = np.exp(X @ betas)

    mem_list = []
    ape_list = []

    for i in range(1, n_main + 1):  # skip constant at index 0
        beta_i = betas[i]
        # MEM: beta * mu_bar
        mu_bar = np.exp(X.mean(axis=0) @ betas)
        mem = beta_i * mu_bar
        # APE: mean of beta * mu_i over all obs
        ape = np.mean(beta_i * mu)
        mem_list.append(mem)
        ape_list.append(ape)

    # Bootstrap 95% CI for APE
    n_boot = 500
    ape_boot = np.zeros((n_boot, n_main))
    rng = np.random.default_rng(42)
    for b in range(n_boot):
        idx = rng.integers(0, len(X), len(X))
        mu_b = np.exp(X[idx] @ betas)
        for i in range(n_main):
            ape_boot[b, i] = np.mean(betas[i + 1] * mu_b)

    ape_ci_lo = np.percentile(ape_boot, 2.5, axis=0)
    ape_ci_hi = np.percentile(ape_boot, 97.5, axis=0)

    return pd.DataFrame({
        "Variable": x_names,
        "MEM": mem_list,
        "APE": ape_list,
        "APE_CI_lo": ape_ci_lo,
        "APE_CI_hi": ape_ci_hi
    })

me_df = marginal_effects_nb(nb2, X_nb2, X_names, len(X_names))
print("Marginal effects (Model 2):")
print(me_df.to_string(index=False))

me_df.to_csv(OUTPUTS / "tables" / "marginal_effects_model2.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

y_pos = np.arange(len(X_names))

ax.scatter(me_df["MEM"], y_pos, marker="o", color="black",
           zorder=5, label="Average ME", s=40)
ax.scatter(me_df["APE"], y_pos + 0.2, marker="^", color="black",
           zorder=5, label="Average PE", s=40)

for i, row in me_df.iterrows():
    ax.plot([row["APE_CI_lo"], row["APE_CI_hi"]],
            [i + 0.2, i + 0.2], color="black", linewidth=1.2)

ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_yticks(y_pos + 0.1)
ax.set_yticklabels(X_names, fontsize=10)
ax.set_xlabel("Marginal Effect", fontsize=11)
ax.set_title(
    "Marginal and Average Partial Effect of Covariates on S&S\n"
    "(Model 2, Negative Binomial)",
    fontsize=11
)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUTS / "figures" / "fig3_marginal_effects.png",
            dpi=300, bbox_inches="tight")
plt.show()
print("Saved fig3_marginal_effects.png")

## Figure 4:  Interaction effect

Replicates Figure 4 from the paper - predicted S&S count as a function of Gini coefficient at five levels of income deprivation (-2 to +2 SD), comparing Model 2 (no interaction) and Model 3 (with interaction).
All other covariates fixed at their means.

In [ ]:
def predict_nb_at_values(model, X_template, gini_idx, income_idx,
                         gini_range, income_levels, has_interaction=False,
                         interaction_idx=None):
    """
    Predict S&S count over a range of Gini values at fixed income levels.
    All other variables held at zero (their standardised mean).
    """
    results = {}
    for inc_level in income_levels:
        preds = []
        for g in gini_range:
            x = X_template.copy()
            x[gini_idx] = g
            x[income_idx] = inc_level
            if has_interaction and interaction_idx is not None:
                x[interaction_idx] = g * inc_level
            preds.append(np.exp(x @ model.params[:-1]))
        results[inc_level] = preds
    return results

gini_range = np.linspace(-2, 2, 100)
income_levels = [-2, -1, 0, 1, 2]
colors = ["#2c7bb6", "#abd9e9", "#ffffbf", "#fdae61", "#d7191c"]

# Template row — all zeros (means after standardisation), with constant
x_template_m2 = np.zeros(X_nb2.shape[1])
x_template_m2[0] = 1  # constant

x_template_m3 = np.zeros(X_nb3.shape[1])
x_template_m3[0] = 1  # constant

# Gini is index 1, income is index 4 (after constant)
gini_idx = 1
income_idx = 4

preds_m2 = predict_nb_at_values(
    nb2, x_template_m2, gini_idx, income_idx, gini_range, income_levels
)
preds_m3 = predict_nb_at_values(
    nb3, x_template_m3, gini_idx, income_idx, gini_range, income_levels,
    has_interaction=True, interaction_idx=9
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, preds, title in zip(
        axes,
        [preds_m2, preds_m3],
        ["Linear (Model 2)", "Interaction (Model 3)"]
):
    for inc_level, color in zip(income_levels, colors):
        ax.plot(gini_range, preds[inc_level], color=color,
                linewidth=2, label=f"{inc_level}")
    ax.set_xlabel("Gini coefficient (SD)", fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.set_xlim(-2, 2)

axes[0].set_ylabel("Predicted S&S", fontsize=11)

# Shared legend
handles = [plt.Line2D([0], [0], color=c, linewidth=2)
           for c in colors]
fig.legend(
    handles, [str(l) for l in income_levels],
    title="Income deprivation (SD)",
    loc="center right",
    bbox_to_anchor=(1.0, 0.5),
    fontsize=9,
    title_fontsize=10
)

plt.suptitle(
    "Effect of Interaction Term on Predicted S&S",
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.savefig(OUTPUTS / "figures" / "fig4_interaction.png",
            dpi=300, bbox_inches="tight")
plt.show()
print("Saved fig4_interaction.png")

## Robustness Check - Table A.2: Stops rate per 1,000 residents

Replicates Table A.2 from the paper. Uses log(stops/1,000 residents) as the outcome variable rather than raw counts, with OLS and SDM.  Confirms results are not sensitive to the choice of outcome measure.

In [ ]:
y_rate = reg_df["log_ss_rate_std"].values

print("Fitting OLS (rate outcome)...")
ols_rate = spreg.OLS(
    y_rate.reshape(-1, 1),
    X_with_fe,
    w=w,
    spat_diag=True,
    moran=True,
    name_y="log_ss_rate",
    name_x=X_names + list(borough_dummies.columns)
)
print(f"OLS (rate) R²: {ols_rate.r2:.3f}")
print(f"OLS (rate) Moran's I: {ols_rate.moran_res[0]:.3f} "
      f"(p={ols_rate.moran_res[2]:.4f})")

print("Fitting SDM (rate outcome)...")
sdm_rate = spreg.ML_Lag(
    y_rate.reshape(-1, 1),
    X_with_fe,
    w=w,
    slx_lags=1,
    spat_diag=True,
    name_y="log_ss_rate",
    name_x=X_names + list(borough_dummies.columns)
)
print(f"SDM (rate) Log-likelihood: {sdm_rate.logll:.3f}")
print(f"SDM (rate) rho: {sdm_rate.rho:.3f}")

table_a2 = build_table2(ols_rate, sdm_rate, X_names)
table_a2.to_csv(OUTPUTS / "tables" / "table_a2_rate_robustness.csv", index=False)
print("\nSaved table_a2_rate_robustness.csv")

## Robustness Check - Table A.3: Westminster excluded

Replicates Table A.3 from the paper. Westminster has neighbourhoods with extremely high inequality and high S&S. Removing it tests whether the main result is driven by this outlier borough.

In [ ]:
reg_df_no_west = reg_df[reg_df["LAD22NM"] != "Westminster"].copy()
print(f"Sample without Westminster: {len(reg_df_no_west)} LSOAs")

# Rebuild borough dummies and spatial weights for this subsample
borough_fe_nw = pd.get_dummies(
    reg_df_no_west["LAD22NM"], drop_first=True, dtype=float
)

reg_gdf_nw = london_gdf.merge(
    reg_df_no_west[["LSOA21CD"]], on="LSOA21CD", how="inner"
)
reg_gdf_nw = (reg_gdf_nw
              .set_index("LSOA21CD")
              .loc[reg_df_no_west["LSOA21CD"]]
              .reset_index())

w_nw = libpysal.weights.Queen.from_dataframe(
    reg_gdf_nw, geom_col="geometry"
)
w_nw.transform = "r"

X_base_nw = reg_df_no_west[X_cols].values
X_with_fe_nw = np.hstack([X_base_nw, borough_fe_nw.values])
y_nw = reg_df_no_west["ss_count"].values

# NB Models 1, 2, 3 without Westminster
gini_nw = reg_df_no_west["gini_housing_std"].values
income_nw = reg_df_no_west["income_score_std"].values

print("Fitting NB models without Westminster...")

X_nb1_nw = sm.add_constant(np.hstack([gini_nw.reshape(-1, 1),
                                      borough_fe_nw.values]))
nb1_nw = NegativeBinomial(y_nw, X_nb1_nw).fit(
    method="bfgs", maxiter=1000, disp=False
)

X_nb2_nw = sm.add_constant(np.hstack([X_base_nw, borough_fe_nw.values]))
nb2_nw = NegativeBinomial(y_nw, X_nb2_nw).fit(
    method="bfgs", maxiter=1000, disp=False
)

interaction_nw = (gini_nw * income_nw).reshape(-1, 1)
X_nb3_nw = sm.add_constant(np.hstack([X_base_nw, interaction_nw,
                                      borough_fe_nw.values]))
nb3_nw = NegativeBinomial(y_nw, X_nb3_nw).fit(
    method="bfgs", maxiter=1000, disp=False
)

table_a3 = build_table3(nb1_nw, nb2_nw, nb3_nw, X_names)
print("\nTable A.3: Without Westminster")
print(table_a3[table_a3["Variable"].isin(
    X_names + ["Gini × Income deprivation", "Observations",
               "Log-likelihood", "AIC"]
)].to_string(index=False))

table_a3.to_csv(OUTPUTS / "tables" / "table_a3_no_westminster.csv", index=False)
print("\nSaved table_a3_no_westminster.csv")


## Robustness Check - S60 vs PACE (extension beyond original paper)

The original paper noted the inability to distinguish between PACE searches (requiring reasonable suspicion) and Section 60 searches (suspicion-less) as a limitation. This analysis addresses that limitation by rerunning the main negative binomial model (Model 2) separately for PACE and S60 counts.  If the inequality-S&S relationship holds in both, this strengthens the finding by showing it is not driven by geographically concentrated S60 authorisations.

In [ ]:
# PACE model
y_pace = reg_df.merge(
    analytical[["LSOA21CD", "ss_count_pace"]], on="LSOA21CD", how="left"
)["ss_count_pace"].fillna(0).astype(int).values

print("Fitting NB Model 2 — PACE searches only...")
nb2_pace = NegativeBinomial(y_pace, X_nb2).fit(
    method="bfgs", maxiter=1000, disp=False
)
print(f"PACE Gini coef: {nb2_pace.params[1]:.3f} "
      f"(p={nb2_pace.pvalues[1]:.4f}), "
      f"IRR: {np.exp(nb2_pace.params[1]):.3f}")

# S60 model
y_s60 = reg_df.merge(
    analytical[["LSOA21CD", "ss_count_s60"]], on="LSOA21CD", how="left"
)["ss_count_s60"].fillna(0).astype(int).values

print("Fitting NB Model 2 — S60 searches only...")
nb2_s60 = NegativeBinomial(y_s60, X_nb2).fit(
    method="bfgs", maxiter=1000, disp=False
)
print(f"S60 Gini coef: {nb2_s60.params[1]:.3f} "
      f"(p={nb2_s60.pvalues[1]:.4f}), "
      f"IRR: {np.exp(nb2_s60.params[1]):.3f}")

# Save S60/PACE comparison table
s60_pace_table = pd.DataFrame({
    "Variable": X_names,
    "All searches coef": [f"{nb2.params[i+1]:.3f}" for i in range(len(X_names))],
    "All searches IRR": [f"{np.exp(nb2.params[i+1]):.3f}" for i in range(len(X_names))],
    "PACE coef": [f"{nb2_pace.params[i+1]:.3f}" for i in range(len(X_names))],
    "PACE IRR": [f"{np.exp(nb2_pace.params[i+1]):.3f}" for i in range(len(X_names))],
    "S60 coef": [f"{nb2_s60.params[i+1]:.3f}" for i in range(len(X_names))],
    "S60 IRR": [f"{np.exp(nb2_s60.params[i+1]):.3f}" for i in range(len(X_names))],
})

print("\nS60 vs PACE comparison (Model 2 coefficients):")
print(s60_pace_table.to_string(index=False))

s60_pace_table.to_csv(
    OUTPUTS / "tables" / "table_s60_pace_robustness.csv", index=False
)
print("\nSaved table_s60_pace_robustness.csv")

## Summary of Moran's I results

In [ ]:
print("=" * 55)
print("Moran's I Summary")
print("=" * 55)
print(f"OLS residuals (count):  I={ols.moran_res[0]:.3f}, "
      f"p={ols.moran_res[2]:.4f} → spatial autocorrelation confirmed")
print(f"SDM residuals (count):  I={mi_sdm.I:.3f}, "
      f"p={mi_sdm.p_sim:.3f} → spatial autocorrelation eliminated")
print(f"OLS residuals (rate):   I={ols_rate.moran_res[0]:.3f}, "
      f"p={ols_rate.moran_res[2]:.4f}")
print("=" * 55)